In [ ]:
# Eval config — injected at submit time by KaggleTrainingAdapter
CONFIG = {{config}}
print('Eval for training run:', CONFIG['training_run_id'])
print('Experiment         :', CONFIG['experiment_name'])
print('Eval data file     :', CONFIG['eval_data_file'])

In [ ]:
import subprocess, sys, re
from pathlib import Path

dataset_slug = CONFIG['experiment_name'] + '-data'
_candidates = list(Path('/kaggle/input').glob(f'**/{dataset_slug}'))
input_base = _candidates[0] if _candidates else Path('/kaggle/input') / dataset_slug
print(f'Dataset mounted at: {input_base}')

# Test CUDA embedding kernel — catches SM-version mismatches early.
_CUDA_KERNEL_TEST = '''
import torch, sys
try:
    w = torch.zeros(100, 8).cuda()
    idx = torch.zeros(2, 3, dtype=torch.long).cuda()
    torch.embedding(w, idx)
    cap = torch.cuda.get_device_capability()
    print(f"CUDA OK gpu={torch.cuda.get_device_name(0)} sm={cap[0]}{cap[1]:02d} torch={torch.__version__}")
except Exception as e:
    cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
    print(f"CUDA FAIL sm={cap[0]}{cap[1]:02d} torch={torch.__version__}: {e}", file=sys.stderr)
    sys.exit(1)
'''

cuda_test = subprocess.run([sys.executable, '-c', _CUDA_KERNEL_TEST], capture_output=True, text=True)
if 'CUDA OK' in cuda_test.stdout:
    print(cuda_test.stdout.strip())
else:
    print('CUDA embedding test failed:', (cuda_test.stderr or cuda_test.stdout).strip()[:400])

    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    m = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', smi.stdout)
    cuda_ver = int(m.group(1)) * 100 + int(m.group(2)) if m else 0

    sm_test = subprocess.run(
        [sys.executable, '-c',
         'import torch; cap=torch.cuda.get_device_capability(); print(cap[0]*100+cap[1])'],
        capture_output=True, text=True,
    )
    sm_ver = int(sm_test.stdout.strip()) if sm_test.stdout.strip().isdigit() else 0
    print(f'CUDA driver={cuda_ver}  GPU SM={sm_ver}')

    if sm_ver < 700:
        cu_tag = 'cu118'
        torch_spec = 'torch==2.2.2'
        print(f'WARNING: P100/Pascal GPU detected (SM={sm_ver}). Pinning to torch==2.2.2+cu118.')
    elif cuda_ver >= 1208 or sm_ver >= 900:
        cu_tag, torch_spec = 'cu128', 'torch'
    elif cuda_ver >= 1204 or sm_ver >= 800:
        cu_tag, torch_spec = 'cu124', 'torch'
    else:
        cu_tag, torch_spec = 'cu121', 'torch'

    print(f'Reinstalling {torch_spec} from {cu_tag} ...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
        torch_spec, '--index-url', f'https://download.pytorch.org/whl/{cu_tag}'
    ], check=True)

    verify = subprocess.run([sys.executable, '-c', _CUDA_KERNEL_TEST], capture_output=True, text=True)
    result = verify.stdout.strip() or verify.stderr.strip()[:400]
    print('After reinstall:', result)
    if 'CUDA OK' not in verify.stdout:
        raise RuntimeError(f'torch reinstall did not fix CUDA kernel compatibility: {result}')

# bitsandbytes + peft may be needed if the checkpoint used QLoRA.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes', 'peft'], check=True)
print('bitsandbytes + peft ready.')

# Force-reinstall the project wheel so the installed package matches the training run.
import os
print('Contents of /kaggle/input:')
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

wheels = list(input_base.glob('*.whl')) or list(input_base.glob('**/*.whl'))
if not wheels:
    raise FileNotFoundError(f'No .whl found under {input_base}')
wheel = wheels[0]
print(f'Installing {wheel.name} (force-reinstall) ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '-q', str(wheel)],
    check=True,
)
print('Done.')

In [ ]:
import subprocess, tarfile
from pathlib import Path

checkpoint_dest = Path('/tmp/checkpoint')
checkpoint_dest.mkdir(parents=True, exist_ok=True)

print(f'Downloading checkpoint from training run: {CONFIG["training_run_id"]} ...')
subprocess.run(
    ['kaggle', 'kernels', 'output', CONFIG['training_run_id'], '-p', str(checkpoint_dest)],
    check=True,
)

archive = checkpoint_dest / 'checkpoint.tar.gz'
if archive.exists():
    print(f'Extracting {archive} ({archive.stat().st_size / 1e6:.1f} MB) ...')
    with tarfile.open(archive) as tf:
        tf.extractall(checkpoint_dest)
    archive.unlink()
    print('Extraction complete.')
else:
    print('No checkpoint.tar.gz found — listing downloaded files:')
    for f in sorted(checkpoint_dest.rglob('*')):
        print(' ', f)

# Locate the HF model directory (contains config.json with model_type).
checkpoint_dir = next(
    (p.parent for p in sorted(checkpoint_dest.rglob('config.json'))
     if '"model_type"' in p.read_text()),
    None,
)
if checkpoint_dir is None:
    raise FileNotFoundError(f'No HF model config.json found under {checkpoint_dest}')
print(f'Checkpoint at: {checkpoint_dir}')

In [ ]:
import shutil
from pathlib import Path

dataset_slug = CONFIG['experiment_name'] + '-data'
_candidates = list(Path('/kaggle/input').glob(f'**/{dataset_slug}'))
input_base = _candidates[0] if _candidates else Path('/kaggle/input') / dataset_slug

data_dst = Path('data')
data_dst.mkdir(exist_ok=True)
eval_src = input_base / CONFIG['eval_data_file']
if not eval_src.exists():
    raise FileNotFoundError(f'Eval data not found: {eval_src}')
shutil.copy(eval_src, data_dst / CONFIG['eval_data_file'])
print(f"Copied {CONFIG['eval_data_file']} ({eval_src.stat().st_size // 1000} KB)")

In [ ]:
import contextlib, io, json, os
from pathlib import Path

os.environ['HF_HOME'] = '/kaggle/working/.hf_cache'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from domain.train.evaluate import PASS_THRESHOLD, evaluate, infer_hf, load_hf_pipeline

print(f'Loading HF pipeline from {checkpoint_dir} ...')
pipe = load_hf_pipeline(str(checkpoint_dir))
infer_fn = lambda prompt: infer_hf(pipe, prompt)  # noqa: E731

eval_path = Path('data') / CONFIG['eval_data_file']
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    exit_code = evaluate(eval_path, infer_fn)

output = buf.getvalue()
print(output)

pct_line = next((l for l in output.splitlines() if l.startswith('Valid:')), '')
if pct_line:
    valid_pct = float(pct_line.split('(')[1].split('%')[0]) / 100
else:
    valid_pct = 1.0 if exit_code == 0 else 0.0

result = {'valid_pct': valid_pct, 'passed': valid_pct >= PASS_THRESHOLD}
Path('/kaggle/working/eval_result.json').write_text(json.dumps(result))
print('Eval result:', result)